<a href="https://colab.research.google.com/github/Aksinhaa/snake-venom-resistance/blob/main/ancient_mitogenome_assembly.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This workshop is a joint collaboration between NCBS - Bangalore, India `(Uma Ramakrishnan, Laura Bertola, Devkant Singha, Vinti Nanda)`; University of Edinburgh UK `(Katerina Guschanski, German Hernandez Alonso)`; IISc - Bangalore, India `(Anubhab Khan, Akancha Sinha)`; IISER Mohali, India `(Kritika M. Garg)`; and Birbal Sahni Institute of Palaeosciences (BSIP) - Lucknow, India `(Niraj Rai)`.
Scripts written by: `Kritika M. Garg`
Notebook compiled by: `Akancha Sinha`

In bioinformatics workflows, multiple tools are used, each with specific dependencies and version requirements. Managing these manually can lead to compatibility issues. Conda is an open-source package and environment management system designed to simplify installing software, managing dependencies, and creating isolated environments for projects. Hence, the first step would be to install miniconda and create a working environment where the required tools can be installed.

In [ ]:
%%bash
# Install Miniconda
wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
bash Miniconda3-latest-Linux-x86_64.sh -b -p /usr/local/miniconda

# Initialize conda
source /usr/local/miniconda/etc/profile.d/conda.sh

# Accept Terms of Service
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

# Add channels
conda config --add channels defaults
conda config --add channels bioconda
conda config --add channels conda-forge

We will install the tools required in the working environment; `mito_env`. For the script to run, you will need to install:

* SEQPREP

* MIA

* PRINSEQ

* BBMAP

* FASTX_TOOLKIT


In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda create -y -n mito_env prinseq bbmap fastx_toolkit

MIA and Seqprep was not available in conda, hence we will clone them from the github repository.

`%%bash` allows the entire cell to run as a Bash shell script
in Google Colab.

`source ... conda.sh` initializes Conda so environments can be activated inside Colab.

`conda activate mito_env` activates the Conda environment containing the required bioinformatics tools.

The git clone commands download the Mapping Iterative Assembler (MIA) and SeqPrep2 repositories from GitHub into the working directory.

In [ ]:
%%bash

source /usr/local/miniconda/etc/profile.d/conda.sh

conda activate mito_env

git clone https://github.com/mpieva/mapping-iterative-assembler.git
git clone https://github.com/jeizenga/SeqPrep2.git

After cloning the tools from Github, we will have to install it in our working environmnet.

`conda activate mito_env` activates the Conda environment where dependencies for MIA are installed.

`cd mapping-iterative-assembler/ `moves into the downloaded MIA source code directory.

`sh bootstrap.sh` prepares the build system by generating the required configuration files.

`./configure, make, and make` install configure, compile, and install the Mapping Iterative Assembler on the system.

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh

conda activate mito_env
cd mapping-iterative-assembler/
sh bootstrap.sh
./configure
make
make install

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh

conda activate mito_env
cd SeqPrep2/
make

`mkdir` command creates new directory

`ls` command lists all the content of a directory

The following command will make separate directories to keep the notbook clean and organised.

In [ ]:
%%bash
mkdir -p /content/{fastq_files,reference}
ls /content/

`wget` command is used to download files from the web directly

We will download the raw FASTQ files from the following link, for the downstream analysis.

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh

conda activate mito_env
cd /content/fastq_files/

wget -nc ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR127/085/SRR12727185/SRR12727185_1.fastq.gz
wget -nc ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR127/085/SRR12727185/SRR12727185_2.fastq.gz


We will also download the reference from NCBI

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
cd /content/reference
wget -O KM659865.1.fasta \
"https://www.ncbi.nlm.nih.gov/search/api/download-sequence/?db=nuccore&id=KM659865.1"

This script automates the mitochondrial genome assembly workflow by processing sequencing reads, mapping them to a reference genome, and iteratively refining the assembly using MIA. It simplifies multiple bioinformatics steps into a single executable pipeline for ancient or modern DNA data analysis.

# The script will :
1. trim adapters
2. remove low complexity reads (repetitive sequences)
3. concatenate merge and unmerged reads
4. convert fastq to fasta (using a quality score)
5. remove duplicate reads
6. run MIA

`source /usr/local/miniconda/etc/profile.d/conda.sh` initializes Conda so that environments and installed tools can be accessed properly in Colab.

`cd /content` changes the working directory to the main Colab workspace where files will be downloaded and executed.

`wget ... mito_assembly_pipeline.sh` downloads the mitochondrial assembly pipeline script from the GitHub repository into the current directory.

`chmod +x mito_assembly_pipeline.sh` gives the script executable permission so it can be run directly from the terminal.

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
cd /content
wget https://raw.githubusercontent.com/Paleogenomics/DNA-Post-Processing/master/mito_assembly_pipeline.sh
chmod +x mito_assembly_pipeline.sh

Following command will allow you to view the `mito_assembly_pipeline.sh` file.

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
cat mito_assembly_pipeline.sh

The `mito_assembly_pipeline.sh ` suggests that few lines of the script needs to be edited to declare the path to the tools, therefore we need to edit the script, before we execute it.


The `sed -i `commands modify the pipeline script by replacing placeholder paths with the correct locations of installed tools and reference files in the Colab environment.


Paths for SeqPrep2, MIA, PRINSEQ, BBMap, FASTX Toolkit, substitution matrices, and the mitochondrial reference genome have to be updated.

The command also corrects the fastq_to_fasta executable path so the script can properly convert FASTQ reads into FASTA format during processing.

`cat mito_assembly_pipeline.sh` displays the edited script, allowing verification that all paths were updated successfully.

In [ ]:
%%bash
sed -i 's#SEQPREP=/path/to/SeqPrep2-master#SEQPREP=/content/SeqPrep2#g' mito_assembly_pipeline.sh
sed -i 's#MIA=/path/to/mia-1.0/src#MIA=/content/mapping-iterative-assembler/src#g' mito_assembly_pipeline.sh
sed -i 's#PRINSEQ=/path/to/prinseq#PRINSEQ=/usr/local/miniconda/envs/mito_env/bin#g' mito_assembly_pipeline.sh
sed -i 's#BBMAP=/path/to/bbmap/sh#BBMAP=/usr/local/miniconda/envs/mito_env/bin#g' mito_assembly_pipeline.sh
sed -i 's#FASTX_TOOLKIT=/path/to/fastx_toolkit-0.0.13.2/src#FASTX_TOOLKIT=/usr/local/miniconda/envs/mito_env/bin#g' mito_assembly_pipeline.sh
sed -i 's#ANCIENT_SUBMAT=/path/to/ancient.submat.txt#ANCIENT_SUBMAT=/content/mapping-iterative-assembler/matrices/ancient.submat.txt#g' mito_assembly_pipeline.sh
sed -i 's#REF=/path/to/mitogenomes/organism.fasta#REF=/content/reference/KM659865.1.fasta#g' mito_assembly_pipeline.sh
sed -i 's#fastq_to_fasta/fastq_to_fasta#fastq_to_fasta#g' mito_assembly_pipeline.sh
cat mito_assembly_pipeline.sh

This command executes the mitochondrial assembly pipeline using paired-end sequencing reads as input. All output files generated during the analysis will be saved with the prefix test_data_cynopterus.



* `source /usr/local/miniconda/etc/profile.d/conda.sh` initializes Conda so the required bioinformatics tools are available during execution.
* `./mito_assembly_pipeline.sh` runs the automated mitochondrial assembly pipeline script.
* The two `.fastq.gz` files are paired-end sequencing reads used as input for mitochondrial genome assembly.
* `test_data_cynopterus` is the output prefix, and all generated files from the assembly process will use this name.


In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
./mito_assembly_pipeline.sh /content/fastq_files/SRR12727185_1.fastq.gz /content/fastq_files/SRR12727185_2.fastq.gz test_data_cynopterus


#In case, if the above step takes too much time to run, kindly download the output files directly from below mentioned  zenodo links. The output files has been provided due to time constraints.

#NOTE: Before downloading the output files, please delete the incomplete files by clicking on the "folder" icon and navigating to the directory containing incomplete generated files, otherwise it may cause error in further steps.

Folders to delete:

1: mia_output

2: seqprep_output

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
mkdir /content/mia_output
cd /content/mia_output

wget https://zenodo.org/records/20104613/files/test_data_cynopterus.name_mito.maln.2
wget https://zenodo.org/records/20104613/files/test_data_cynopterus.name_mito.maln.F.inputfornext.txt
wget https://zenodo.org/records/20104613/files/test_data_cynopterus.name_mito.maln.F.mia_consensus.fasta
wget https://zenodo.org/records/20104613/files/test_data_cynopterus.name_mito.maln.F.mia_coverage_per_site.txt
wget https://zenodo.org/records/20104613/files/test_data_cynopterus.name_mito.maln.F.mia_stats.txt


In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
mkdir /content/seqprep_output
cd /content/seqprep_output

wget https://zenodo.org/records/20104613/files/test_data_cynopterus_all_trimmed.complexity_filtered.fasta.gz
wget https://zenodo.org/records/20104613/files/test_data_cynopterus_R1_unmerged.fastq.gz
wget https://zenodo.org/records/20104613/files/test_data_cynopterus_R2_unmerged.fastq.gz
wget https://zenodo.org/records/20104613/files/test_data_cynopterus_readable_alignment.txt
wget https://zenodo.org/records/20104613/files/test_data_cynopterus_unmerged_combined.complexity_filtered.fastq.gz
wget https://zenodo.org/records/20104613/files/test_data_cynopterus_unmerged_combined.fastq.gz



To use the create_fasta_from_mia.py script you'll need to first create a .41 file. You can do that this way



In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
ma -M /content/mia_output/test_data_cynopterus.name_mito.maln.2 -f 41 > cynopterus_assembly.41

This script will create a consensus FASTA file from the output of MIA. It will also filter sequences based on coverage, percentage agreement, and quality.

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
wget -O create_fasta_from_mia.py https://raw.githubusercontent.com/aersoares81/mia-helper-scripts/main/create_fasta_from_mia.py

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
python create_fasta_from_mia.py -m mia_output/test_data_cynopterus.name_mito.maln.F.inputfornext.txt -c 10 -o mt_genome_10x